In [1]:
from matplotlib import pyplot as plt
import pandas as pd
import keras
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from tqdm import tqdm
from PIL import Image
import cv2
import numpy as np
from scipy import stats

%matplotlib qt

In [2]:
# ============================================================
# CHECK FINE-TUNING INPUT FOLDER
# ============================================================

from pathlib import Path
import numpy as np
import cv2

input_dir = Path("Fine Tuning/imageandmask1")

if not input_dir.exists():
    raise FileNotFoundError(f"Input folder not found: {input_dir.resolve()}")

files = sorted([p for p in input_dir.iterdir() if p.is_file()])

print(f"Input folder: {input_dir.resolve()}")
print(f"Total files found: {len(files)}")

for file in files[:10]:
    print(file.name)

Input folder: C:\Users\gabri\SegmentEveryForam\Fine Tuning\Imageandmask1
Total files found: 102
U1559C_1H_1PAL_MUDLINE_G_bulloides_img001_image.jpeg
U1559C_1H_1PAL_MUDLINE_G_bulloides_img001_mask.png
U1559C_1H_1PAL_MUDLINE_G_ruber_img001_image.jpeg
U1559C_1H_1PAL_MUDLINE_G_ruber_img001_mask.png
U1559C_1H_1PAL_MUDLINE_G_trunc_img001_image.jpeg
U1559C_1H_1PAL_MUDLINE_G_trunc_img001_mask.png
U1559C_1H_1W_75-77_G_bulloides_img001_image.jpeg
U1559C_1H_1W_75-77_G_bulloides_img001_mask.png
U1559C_1H_1W_75-77_G_ruber_img001_image.jpeg
U1559C_1H_1W_75-77_G_ruber_img001_mask.png


In [3]:
# Identify image and mask files using the naming convention
image_files = sorted([
    p for p in files
    if "image" in p.stem.lower()
    and "mask" not in p.stem.lower()
])

mask_files = sorted([
    p for p in files
    if "mask" in p.stem.lower()
])

print(f"Image files identified: {len(image_files)}")
print(f"Mask files identified: {len(mask_files)}")

Image files identified: 51
Mask files identified: 51


In [4]:
# ============================================================
# VALIDATE MASK VALUES AND IMAGE–MASK DIMENSIONS
# ============================================================

print("Unique values found in masks:\n")

all_mask_values = set()

for mask_path in mask_files:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        print(f"Could not read: {mask_path.name}")
        continue

    unique_values = np.unique(mask)
    all_mask_values.update(unique_values.tolist())

    print(
        f"{mask_path.name}: "
        f"shape={mask.shape}, "
        f"dtype={mask.dtype}, "
        f"values={unique_values}"
    )

print("\nAll mask values across dataset:")
print(sorted(all_mask_values))

Unique values found in masks:

U1559C_1H_1PAL_MUDLINE_G_bulloides_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_1PAL_MUDLINE_G_ruber_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_1PAL_MUDLINE_G_trunc_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_1W_75-77_G_bulloides_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_1W_75-77_G_ruber_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_1W_75-77_G_ruber_img002_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_CC_PAL_G_bulloides_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_CC_PAL_G_ruber_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_CC_PAL_G_ruber_img002_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_CC_PAL_G_trunc_img001_mask.png: shape=(1460, 1936), dtype=uint8, values=[0 1 2]
U1559C_1H_CC_PAL_G_trunc_img002_m

In [5]:
# ============================================================
# VALIDATE IMAGE–MASK PAIRS
# ============================================================

from pathlib import Path
import cv2

input_dir = Path("Fine Tuning/Imageandmask1")

image_files = sorted([
    p for p in input_dir.iterdir()
    if p.is_file()
    and "image" in p.stem.lower()
    and "mask" not in p.stem.lower()
])

mask_files = sorted([
    p for p in input_dir.iterdir()
    if p.is_file()
    and "mask" in p.stem.lower()
])

# Create lookup dictionaries using the shared filename prefix
image_lookup = {
    p.stem.lower().replace("_image", ""): p
    for p in image_files
}

mask_lookup = {
    p.stem.lower().replace("_mask", ""): p
    for p in mask_files
}

image_keys = set(image_lookup)
mask_keys = set(mask_lookup)

missing_masks = sorted(image_keys - mask_keys)
missing_images = sorted(mask_keys - image_keys)
matched_keys = sorted(image_keys & mask_keys)

print(f"Matched image–mask pairs: {len(matched_keys)}")
print(f"Images without masks: {len(missing_masks)}")
print(f"Masks without images: {len(missing_images)}")

if missing_masks:
    print("\nImages without matching masks:")
    for key in missing_masks:
        print("•", image_lookup[key].name)

if missing_images:
    print("\nMasks without matching images:")
    for key in missing_images:
        print("•", mask_lookup[key].name)

# Check dimensions
dimension_errors = []

for key in matched_keys:
    image_path = image_lookup[key]
    mask_path = mask_lookup[key]

    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if image is None:
        dimension_errors.append(
            f"Could not read image: {image_path.name}"
        )
        continue

    if mask is None:
        dimension_errors.append(
            f"Could not read mask: {mask_path.name}"
        )
        continue

    if image.shape[:2] != mask.shape[:2]:
        dimension_errors.append(
            f"{image_path.name}: image={image.shape[:2]}, "
            f"mask={mask.shape[:2]}"
        )

print(f"\nDimension mismatches: {len(dimension_errors)}")

for error in dimension_errors:
    print("•", error)

if (
    len(matched_keys) == 51
    and not missing_masks
    and not missing_images
    and not dimension_errors
):
    print("\nDataset validation passed.")
    print("The dataset is ready for patch creation.")
else:
    print("\nDataset validation found problems that should be fixed before patchifying.")

Matched image–mask pairs: 51
Images without masks: 0
Masks without images: 0

Dimension mismatches: 0

Dataset validation passed.
The dataset is ready for patch creation.


In [10]:
from pathlib import Path
import os

input_dir = Path("Fine Tuning/Imageandmask1")
patch_dir = Path("Fine Tuning/Patch_output1")

# Add the trailing path separator required by the function
input_dir_for_seg = str(input_dir.resolve()) + os.sep

image_dir, mask_dir = seg.patchify_training_data(
    input_dir_for_seg,
    str(patch_dir.resolve())
)

print("Image patches:", image_dir)
print("Mask patches:", mask_dir)

100%|██████████| 51/51 [00:28<00:00,  1.80it/s]

Image patches: C:\Users\gabri\SegmentEveryForam\Fine Tuning\Patch_output1\Patches\images
Mask patches: C:\Users\gabri\SegmentEveryForam\Fine Tuning\Patch_output1\Patches\labels


In [17]:
# ============================================================
# VISUALIZE RANDOM TRAINING PATCHES
# ============================================================

from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

image_patch_dir = Path(image_dir)
mask_patch_dir = Path(mask_dir)

image_patches = sorted(image_patch_dir.glob("*"))
mask_patches = sorted(mask_patch_dir.glob("*"))

print(f"Image patches: {len(image_patches)}")
print(f"Mask patches : {len(mask_patches)}")

# Display 5 random patch pairs
indices = random.sample(range(len(image_patches)), 5)

fig, axes = plt.subplots(len(indices), 2, figsize=(8, 18))

for row, idx in enumerate(indices):

    img = cv2.imread(str(image_patches[idx]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(
        str(mask_patches[idx]),
        cv2.IMREAD_UNCHANGED
    )

    axes[row,0].imshow(img)
    axes[row,0].set_title(image_patches[idx].name)
    axes[row,0].axis("off")

    axes[row,1].imshow(mask, cmap="viridis", vmin=0, vmax=2)
    axes[row,1].set_title(mask_patches[idx].name)
    axes[row,1].axis("off")

plt.tight_layout()
plt.show()

Image patches: 7140
Mask patches : 7140


In [18]:
print(len(image_patches))

7140


In [19]:
# ============================================================
# CHECK PATCH PAIRING AND MASK CLASS DISTRIBUTION
# ============================================================

from pathlib import Path
import cv2
import numpy as np

image_patch_dir = Path(image_dir)
mask_patch_dir = Path(mask_dir)

image_patches = sorted(image_patch_dir.glob("*"))
mask_patches = sorted(mask_patch_dir.glob("*"))

if len(image_patches) != len(mask_patches):
    raise ValueError(
        f"Patch count mismatch: "
        f"{len(image_patches)} images vs {len(mask_patches)} masks"
    )

class_counts = {
    0: 0,  # background
    1: 0,  # foram interior
    2: 0   # boundary
}

empty_patches = 0
invalid_masks = []

for mask_path in mask_patches:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        invalid_masks.append(mask_path.name)
        continue

    values, counts = np.unique(mask, return_counts=True)

    if not set(values).issubset({0, 1, 2}):
        invalid_masks.append(
            f"{mask_path.name}: values={values.tolist()}"
        )

    for value, count in zip(values, counts):
        if int(value) in class_counts:
            class_counts[int(value)] += int(count)

    if np.all(mask == 0):
        empty_patches += 1

total_pixels = sum(class_counts.values())

print(f"Image patches: {len(image_patches)}")
print(f"Mask patches:  {len(mask_patches)}")
print(f"All-background patches: {empty_patches}")
print(
    f"All-background percentage: "
    f"{100 * empty_patches / len(mask_patches):.2f}%"
)

print("\nPixel class distribution:")

for class_id, class_name in [
    (0, "Background"),
    (1, "Foram interior"),
    (2, "Boundary")
]:
    percentage = 100 * class_counts[class_id] / total_pixels

    print(
        f"{class_id} — {class_name}: "
        f"{class_counts[class_id]:,} pixels "
        f"({percentage:.2f}%)"
    )

print(f"\nInvalid masks: {len(invalid_masks)}")

for problem in invalid_masks[:10]:
    print("•", problem)

Image patches: 7140
Mask patches:  7140
All-background patches: 1593
All-background percentage: 22.31%

Pixel class distribution:
0 — Background: 424,394,160 pixels (90.70%)
1 — Foram interior: 35,933,553 pixels (7.68%)
2 — Boundary: 7,599,327 pixels (1.62%)

Invalid masks: 0


In [20]:
train_dataset, val_dataset, test_dataset = seg.create_train_val_test_data(
    image_dir,
    mask_dir,
    augmentation=True
)

In [23]:
from pathlib import Path

for model in [
    "models/seg_model.keras",
    "models/seg_model_smooth_labels.keras"
]:
    path = Path(model)

    print(path.name)
    print(f"Size: {path.stat().st_size/1024/1024:.2f} MB\n")

seg_model.keras
Size: 24.93 MB

seg_model_smooth_labels.keras
Size: 24.93 MB



In [24]:
from pathlib import Path
import hashlib

def sha256_file(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(block)

    return hasher.hexdigest()


model_paths = [
    Path("models/seg_model.keras"),
    Path("models/seg_model_smooth_labels.keras")
]

for path in model_paths:
    print(path.name)
    print(sha256_file(path))
    print()

seg_model.keras
fd680909cc02a3ed954c7e7584f7da6ff4d4f4975d912eb5afa74166b22ac92e

seg_model_smooth_labels.keras
126b9b4fe299ae1abbfb8d9a6249704b4b0d2e966f0ccdfd41452f1a7d8e323d



In [25]:
# ============================================================
# FINE-TUNE EXISTING U-NET
# Base model: seg_model_smooth_labels.keras
# ============================================================

from pathlib import Path

base_model_path = Path("models/seg_model_smooth_labels.keras")

if not base_model_path.exists():
    raise FileNotFoundError(
        f"Base model not found: {base_model_path.resolve()}"
    )

model = seg.create_and_train_model(
    train_dataset,
    val_dataset,
    test_dataset,
    model_file=str(base_model_path),
    epochs=30
)

Epoch 1/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1202s 8s/step - accuracy: 0.8818 - loss: 0.3661 - val_accuracy: 0.9926 - val_loss: 0.2401
Epoch 2/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1243s 9s/step - accuracy: 0.9909 - loss: 0.2302 - val_accuracy: 0.9923 - val_loss: 0.2261
Epoch 3/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 23885s 168s/step - accuracy: 0.9913 - loss: 0.2252 - val_accuracy: 0.9927 - val_loss: 0.2226
Epoch 4/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1048s 7s/step - accuracy: 0.9916 - loss: 0.2241 - val_accuracy: 0.9925 - val_loss: 0.2213
Epoch 5/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1030s 7s/step - accuracy: 0.9916 - loss: 0.2244 - val_accuracy: 0.9927 - val_loss: 0.2213
Epoch 6/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1071s 7s/step - accuracy: 0.9915 - loss: 0.2244 - val_accuracy: 0.9930 - val_loss: 0.2208
Epoch 7/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1054s 7s/step - accuracy: 0.9917 - loss: 0.2238 - val_accuracy: 0.9933 - val_loss: 0.2208
Epoch 8/30
143/143 ━━━━━━━━━━━━━━━━━━━━ 1078s 8s/step - accuracy: 0.9918 - loss: 0.2234

evaluating model: 100%|██████████| 34/34 [00:22<00:00,  1.52it/s]


Test loss: 0.2216
Test accuracy: 0.9935
Mean IoU: 0.8981
  background IoU: 0.9958
  grain IoU: 0.9549
  boundary IoU: 0.7437


In [26]:
# ============================================================
# SAVE FINE-TUNED MODEL
# ============================================================

from pathlib import Path

output_dir = Path("models")
output_dir.mkdir(exist_ok=True)

model_path = output_dir / "seg_model_foram_v1_30epochs.keras"

model.save(model_path)

print(f"Fine-tuned model saved to:\n{model_path.resolve()}")

Fine-tuned model saved to:
C:\Users\gabri\SegmentEveryForam\models\seg_model_foram_v1_30epochs.keras


In [29]:
# ============================================================
# EVALUATE FINE-TUNED MODEL
# ============================================================

results = seg.evaluate_model(
    model,
    test_dataset
)

evaluating model: 100%|██████████| 34/34 [01:12<00:00,  2.13s/it]


Test loss: 0.2218
Test accuracy: 0.9935
Mean IoU: 0.8981
  background IoU: 0.9958
  grain IoU: 0.9549
  boundary IoU: 0.7437


In [30]:
from pathlib import Path
import json
from datetime import datetime

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

results_to_save = {
    "model_name": "seg_model_foram_v1_30epochs.keras",
    "base_model": "seg_model_smooth_labels.keras",
    "training_images": 51,
    "training_patches": 7140,
    "epochs": 30,
    "training_date": datetime.now().strftime("%Y-%m-%d"),
    "metrics": results
}

results_path = model_dir / "seg_model_foram_v1_30epochs_metrics.json"

with open(results_path, "w") as f:
    json.dump(results_to_save, f, indent=4)

print(f"Saved metrics to:\n{results_path.resolve()}")

Saved metrics to:
C:\Users\gabri\SegmentEveryForam\models\seg_model_foram_v1_30epochs_metrics.json
